# Exploración — Liga Profesional Argentina (Sofascore)

**Plataforma de Datos & Scouting** · @datafutbol_ar

Fuente: `ScraperFC.Sofascore` (id 155). Granularidad: **1 fila por jugador** de la
liga-temporada (rating, defensa, pases… 114 columnas). Sirve para scouting durante el receso.

> Confirmado 23/5: la temporada **2026 (en curso)** está disponible. La función
> `stats_liga()` ya cachea a parquet, así que esto se baja una vez y después lee del disco.

## Setup

In [1]:
import sys
sys.path.insert(0, r'D:\PROYECTOS_venv\02_PROYECTOS\01_Python\datafutbol_ar\plataforma_scouting')
from ingest.sofascore_loader import temporadas, stats_liga
import pandas as pd
pd.set_option('display.max_columns', None)

## 1. Temporadas disponibles

In [2]:
temporadas('Argentina Liga Profesional')   # {temporada: id}

Running


{'2026': 87913,
 '2025': 70268,
 '2024': 57478,
 '2023': 47647,
 '2022': 41884,
 '2021': 37231,
 '19/20': 24239,
 '18/19': 18113,
 '17/18': 13950,
 '16/17': 12117,
 '2016': 11237,
 '2015': 9651,
 '2014': 8338,
 '13/14': 6455,
 '12/13': 5103,
 '11/12': 3613,
 '10/11': 2887,
 '09/10': 2323,
 '08/09': 1636}

## 2. Cargar la temporada

Primera vez scrapea y guarda; después lee del cache. Para re-bajar la temporada en
curso (que cambia cada fecha) usá `refrescar=True`.

In [3]:
df = stats_liga('2026', 'Argentina Liga Profesional', accumulation='per90')
print(df.shape)   # (jugadores, columnas)

[API] scrapeando Argentina Liga Profesional 2026 (per90)...
[API] guardado stats_argentina_liga_profesional_2026_per90.parquet: 821 jugadores x 114 cols
(821, 114)


## 3. Identificar las columnas clave

Son 114 columnas con nombres crudos de Sofascore (inglés, camelCase). Antes de analizar
necesitamos ubicar las de **identificación** (nombre, equipo, posición) y las de
**volumen** (minutos, rating). Truco: las columnas de **texto** suelen ser las de identificación.

In [4]:
# Columnas de texto (suelen ser nombre / equipo / posición)
texto = df.select_dtypes(include='object').columns.tolist()
print('TEXTO:', texto)

# Columnas típicas de identificación / volumen / rating
claves = [c for c in df.columns if any(k in c.lower() for k in
          ['name','team','position','minut','appear','rating','goal','assist'])]
print('CLAVE:', claves)

TEXTO: ['player', 'team']
CLAVE: ['accurateOppositionHalfPasses', 'appearances', 'assists', 'countRating', 'errorLeadToGoal', 'expectedAssists', 'expectedGoals', 'freeKickGoal', 'goalConversionPercentage', 'goalKicks', 'goals', 'goalsAssistsSum', 'goalsConceded', 'goalsConcededInsideTheBox', 'goalsConcededOutsideTheBox', 'goalsFromInsideTheBox', 'goalsFromOutsideTheBox', 'goalsPrevented', 'headedGoals', 'leftFootGoals', 'minutesPlayed', 'ownGoals', 'passToAssist', 'penaltyGoals', 'rating', 'rightFootGoals', 'totalAttemptAssist', 'totalOppositionHalfPasses', 'totalRating', 'totwAppearances', 'team', 'team id']


C:\Users\lucas\AppData\Local\Temp\ipykernel_32184\2539212779.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  texto = df.select_dtypes(include='object').columns.tolist()


In [5]:
# Ojeamos las primeras columnas de texto para ver qué traen
df[texto[:6]].head()

,player,team
0,Jherson Mosquera,Newell's Old Boys
1,Leandro Paredes,Boca Juniors
2,Josué Reinatti,Newell's Old Boys
3,Marcelo Agustín Miño,Barracas Central
4,Sebastián Villa,Independiente Rivadavia


## 4. Próximo paso

Cuando corras la celda 3, fijate los nombres EXACTOS de:
- columna de **nombre** del jugador,
- columna de **equipo**,
- columna de **minutos** jugados,
- columna de **rating**.

Con eso armamos: (a) filtro por minutos mínimos, (b) primer ranking útil (ej. **sub-23
con mejor rating** o **top defensores por intercepciones/90**), y (c) un mapeo de las
columnas clave a español para que el análisis sea legible. Pasame esos 4 nombres y seguimos.